# 03 Matching Comparison — binary SKU matching benchmark

Эта тетрадка начинается после ручной разметки. На входе у нас есть CSV с парами товаров и твоими метками: тот же это базовый товар или другой товар.

Теперь benchmark работает как простая forced binary evaluation:

- считает `score` для rule-based, bi-encoder, cross-encoder и выбранных reranker-моделей;
- строит binary target `same_base_product` (`exact_duplicate` и legacy `same_product_different_pack` = 1, `different_product` = 0);
- на `dev` подбирает один `threshold_same` для нескольких стратегий;
- применяет выбранные на `dev` пороги к `test` без подбора на test;
- сохраняет компактные CSV в `artifacts/reports/`.

Правило предсказания одно: `predicted_binary = 1`, если `score >= threshold_same`, иначе `0`.

Фасовка и multipack не являются отдельным ML-классом: они разбираются после модели deterministic правилами.


## Мини-словарь перед запуском

`same_base_product` — бинарный таргет для модели: `1`, если это тот же базовый товар, и `0`, если это другой товар.

`threshold_same` — единственный порог: `score >= threshold_same` означает `same_base_product=1`, ниже порога — `different_product=0`.

`false merge` — дорогая ошибка: настоящий `different_product` предсказан как тот же базовый товар.

`false split` — более дешёвая ошибка: настоящий same-base товар предсказан как `different_product`.

`threshold_max_f1` — порог с максимальным обычным F1 на `dev`.

`threshold_cost_sensitive` — порог с минимальной ценой ошибки на `dev`: `5 * false_merge + 1 * false_split`.

Weighted-метрики считаются только если есть надёжный объём продаж по SKU A/B. Основной вес — именно объём продаж, не выручка.


## Как устроена проверка

Мы делим размеченные пары на две части:

- `dev` — часть для подбора `threshold_same`, bucket cutoffs и стратегий threshold.
- `test` — отложенная часть для честной проверки. На ней запрещено выбирать threshold, bucket cutoffs или модель.

Для каждого method считаются стратегии `threshold_max_f1` и `threshold_cost_sensitive`. Если доступны объёмы продаж, добавляются `threshold_max_weighted_f1` и `threshold_weighted_cost`, а также breakdown по bucket объёма продаж: `zero / low / medium / high`.


## Блок кода 1. Подготовка окружения

Эта ячейка подключает библиотеки, находит корень проекта и импортирует нужные функции из `research/dedup`.

Если здесь ошибка, чаще всего причина простая: тетрадка запущена не из папки проекта или не установлен пакет для ноутбуков.


In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib.util
import os
import sys
import time
from typing import Any

from IPython.display import display
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.dedup import (
    CROSS_ENCODER_BACKEND,
    SENTENCE_TRANSFORMER_BACKEND,
    TRANSFORMERS_AUTO_MODEL_BACKEND,
    BiEncoderMatcher,
    BinaryThresholdConfig,
    FusionConfig,
    ModelManager,
    POLZA_EMBEDDING_BACKEND,
    RuleBasedMatcher,
    calibrate_and_evaluate_methods,
    detect_sales_volume_columns,
    same_base_product_target,
    write_binary_threshold_reports,
)
from research.dedup.matchers.bi_encoder import BiEncoderConfig

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)


## Блок кода 2. Настройки тетрадки

Здесь задаются пути к файлам и основные параметры проверки.

Самое важное:

- `LABELING_PATH` — файл с ручной разметкой.
- `RUN_BI_ENCODER` — запускать ли модель для сравнения текстов.
- `BI_ENCODER_MODEL` — alias из `research.dedup.model_registry` или прямой model id, например `openai/text-embedding-3-small`.
- `DEDUP_BI_ENCODER_BACKEND=polza_embedding` — только для нового прямого Polza model id, которого ещё нет в registry; известные Polza ids распознаются автоматически.
- `POLZA_API_KEY` или `POLZA_AI_API_KEY` — ключ для online-моделей Polza.ai.
- `DEDUP_MODEL_CACHE_DIR` — куда локально складываются скачанные local-модели; по умолчанию `research/dedup/models/`.
- `DEDUP_MODEL_LOCAL_ONLY=1` — offline-режим: не скачивать модель, а брать только уже лежащую в кэше.
- `DEV_FRACTION` — какая часть размеченных пар идёт на подбор порогов.
- `FP_COST` / `FN_COST` — цена false merge и false split для cost-sensitive стратегии.

Если в разметке уже есть `sales_volume_a` / `sales_volume_b`, notebook использует их. Если нет, он попробует подтянуть `Продажи, шт` из `mpstats_products` по `raw_record_id` (`marketplace + sku`). Если это не получается надёжно, weighted-метрики отключаются и используется `pair_weight=1`.


In [ ]:
DATA_DIR = PROJECT_ROOT / "research" / "dedup" / "data"
REPORTS_DIR = PROJECT_ROOT / "artifacts" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
LABELING_PATH = DATA_DIR / "labeling_sauces.csv"

SOURCE_LABELS = ["exact_duplicate", "same_product_different_pack", "different_product"]
CATEGORY_ALIASES = ["Соусы", "Соус"]
PRODUCTS_TABLE = "mpstats_products"
SALES_VOLUME_COL = "Продажи, шт"
SALES_VOLUME_JOIN_ENABLED = os.environ.get("DEDUP_ENABLE_SALES_VOLUME_JOIN", "1") == "1"

MODEL_MANAGER = ModelManager()
RUN_BI_ENCODER = os.environ.get("DEDUP_RUN_BI_ENCODER", "1") == "1"
BI_ENCODER_MODEL = os.environ.get("DEDUP_BI_ENCODER_MODEL", "bi_encoder_e5_small")
BI_ENCODER_MODEL_BACKEND = os.environ.get("DEDUP_BI_ENCODER_BACKEND", "").strip() or None
BI_ENCODER_MODEL_SPEC = MODEL_MANAGER.resolve_embedding_model(BI_ENCODER_MODEL, backend=BI_ENCODER_MODEL_BACKEND)
RANDOM_STATE = int(os.environ.get("DEDUP_EVAL_RANDOM_STATE", "42"))
DEV_FRACTION = float(os.environ.get("DEDUP_EVAL_DEV_FRACTION", "0.60"))
FP_COST_VALUE = float(os.environ.get("DEDUP_FP_COST", "5"))
FN_COST_VALUE = float(os.environ.get("DEDUP_FN_COST", "1"))
THRESHOLD_CONFIG = BinaryThresholdConfig(fp_cost=FP_COST_VALUE, fn_cost=FN_COST_VALUE)

print(f"Labeling path: {LABELING_PATH}")
print(f"Reports dir: {REPORTS_DIR}")
print(f"Run bi-encoder: {RUN_BI_ENCODER}")
print(f"Bi-encoder model alias/input: {BI_ENCODER_MODEL}")
print(f"Bi-encoder model id: {BI_ENCODER_MODEL_SPEC.model_name}")
print(f"Bi-encoder backend: {BI_ENCODER_MODEL_SPEC.backend}")
if BI_ENCODER_MODEL_SPEC.backend == POLZA_EMBEDDING_BACKEND:
    print(f"Polza base URL: {MODEL_MANAGER.polza_base_url}")
print(f"Model cache dir: {MODEL_MANAGER.cache_dir}")
print(f"Local-only model loading: {MODEL_MANAGER.local_files_only}")
print(f"Dev fraction: {DEV_FRACTION:.0%}; random_state={RANDOM_STATE}")
print(f"Cost weights: FP_COST={FP_COST_VALUE:g}; FN_COST={FN_COST_VALUE:g}")
print(f"Sales volume join enabled: {SALES_VOLUME_JOIN_ENABLED}")


## Блок кода 3. Загрузка и первичная проверка разметки

Эта ячейка читает `labeling_sauces.csv`, проверяет колонку `label`, убирает `uncertain` из расчёта метрик и делит строки на `dev` и `test`.

В выводе нужно смотреть:

- сколько всего строк в разметке;
- сколько строк реально попало в метрики;
- сколько строк отброшено как `uncertain`;
- как классы распределились между `dev` и `test`.

Если классов в `test` очень мало, итоговые цифры будут шумными.


In [ ]:
def load_labeled_pairs(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not path.exists():
        status = pd.DataFrame([
            {
                "status": "missing_labeling_file",
                "message": f"Файл {path} пока не найден. Выполните 02_labeling_dataset.ipynb и заполните label.",
                "rows_total": 0,
                "rows_usable_for_metrics": 0,
                "rows_ignored_without_binary_target": 0,
            }
        ])
        return pd.DataFrame(), status

    frame = pd.read_csv(path)
    if "label" not in frame.columns and "same_base_product" not in frame.columns:
        status = pd.DataFrame([
            {
                "status": "missing_target_columns",
                "message": "В файле нет ни label, ни same_base_product. Перегенерируйте labeling dataset из notebook-2.",
                "rows_total": len(frame),
                "rows_usable_for_metrics": 0,
                "rows_ignored_without_binary_target": len(frame),
            }
        ])
        return pd.DataFrame(), status

    target = same_base_product_target(frame)
    labeled = frame[target.notna()].copy()
    labeled["same_base_product"] = target[target.notna()].astype(int).to_numpy()
    if "label" in labeled.columns:
        labeled["label"] = labeled["label"].fillna("").astype(str).str.strip()
    ignored_count = int(len(frame) - len(labeled))
    status_name = "ready" if not labeled.empty else "empty_or_not_reviewed_yet"
    message = (
        "Gold-set готов для cost-sensitive calibration."
        if not labeled.empty
        else "Binary target пока не заполнен: метрики ниже будут заглушками, notebook не падает."
    )
    status = pd.DataFrame([
        {
            "status": status_name,
            "message": message,
            "rows_total": len(frame),
            "rows_usable_for_metrics": len(labeled),
            "rows_ignored_without_binary_target": ignored_count,
        }
    ])
    return labeled.reset_index(drop=True), status


def add_stratified_eval_split(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame.assign(eval_split=pd.Series(dtype="string"))
    parts: list[pd.DataFrame] = []
    for _, group in frame.groupby("same_base_product", sort=False):
        shuffled = group.sample(frac=1.0, random_state=RANDOM_STATE)
        if len(shuffled) == 1:
            dev = shuffled.copy()
            dev["eval_split"] = "dev"
            parts.append(dev)
            continue
        dev_count = int(round(len(shuffled) * DEV_FRACTION))
        dev_count = min(max(1, dev_count), len(shuffled) - 1)
        dev = shuffled.iloc[:dev_count].copy()
        test = shuffled.iloc[dev_count:].copy()
        dev["eval_split"] = "dev"
        test["eval_split"] = "test"
        parts.extend([dev, test])
    return pd.concat(parts).sort_index().reset_index(drop=True)


labeled_pairs, labeling_status = load_labeled_pairs(LABELING_PATH)
labeled_pairs = add_stratified_eval_split(labeled_pairs)

display(labeling_status)
if labeled_pairs.empty:
    display(pd.DataFrame(columns=["same_base_product", "pairs"]))
else:
    display(labeled_pairs["same_base_product"].map({1: "same_base_product", 0: "different_product"}).value_counts().rename_axis("target").reset_index(name="pairs"))
    display(pd.crosstab(labeled_pairs["eval_split"], labeled_pairs["same_base_product"].map({1: "same_base_product", 0: "different_product"})))
    if "label" in labeled_pairs.columns:
        display(labeled_pairs["label"].value_counts().rename_axis("source_label").reset_index(name="pairs"))


## Блок кода 4. Список методов, которые будем сравнивать

Эта ячейка создаёт методы сравнения пар и показывает, доступны ли они в текущем окружении.

Важно смотреть на строку `bi_encoder_zero_shot`:

- `available=True` означает, что пакет найден;
- `available=False` означает, что модельный способ будет пропущен.

Если метод пропущен, это не портит rule-based проверку, но сравнение с моделью будет неполным.


In [ ]:
matchers = [RuleBasedMatcher()]
if RUN_BI_ENCODER:
    matchers.append(BiEncoderMatcher(BiEncoderConfig(model_name=BI_ENCODER_MODEL, model_backend=BI_ENCODER_MODEL_BACKEND)))

method_status = []
for matcher in matchers:
    status = matcher.status()
    method_status.append({"method": matcher.name, "available": status.available, "status": status.message})

method_status_df = pd.DataFrame(method_status)
display(method_status_df)


## Блок кода 5. Вспомогательные функции для scoring

Эта ячейка не выбирает пороги. Она только задаёт общий способ посчитать `score` для пары и собрать score-таблицу.

Пороговая логика живёт ниже в финальном binary threshold benchmark block.


In [ ]:
def _score_matcher(matcher: Any, pairs: pd.DataFrame) -> tuple[list[float], str]:
    if pairs.empty:
        return [], "skipped_empty_gold_set"
    row_objects = [row for _, row in pairs.iterrows()]
    score_batch = getattr(matcher, "score_batch", None)
    if callable(score_batch):
        scores = score_batch(row_objects)
    else:
        scores = [matcher.score(row) for row in row_objects]
    if scores and all(pd.isna(score) for score in scores):
        return scores, matcher.status().message
    return scores, "ready"


def _with_benchmark_pair_key(frame: pd.DataFrame) -> pd.DataFrame:
    output = frame.copy()
    if {"raw_record_id_a", "raw_record_id_b"}.issubset(output.columns):
        left_values = output["raw_record_id_a"].astype(str)
        right_values = output["raw_record_id_b"].astype(str)
    else:
        left_values = output.get("title_a", pd.Series([""] * len(output))).astype(str)
        right_values = output.get("title_b", pd.Series([""] * len(output))).astype(str)
    output["benchmark_pair_key"] = [
        " || ".join(sorted([left, right]))
        for left, right in zip(left_values, right_values, strict=False)
    ]
    return output


def _score_frame(method: str, frame: pd.DataFrame, scores: list[float], *, benchmark_source: str) -> pd.DataFrame:
    output = _with_benchmark_pair_key(frame)
    output["method"] = method
    output["score"] = scores
    output["benchmark_source"] = benchmark_source
    return output


## Блок кода 6. Подсчёт сходства для всех пар

Эта ячейка прогоняет каждый метод по размеченным парам и сохраняет численные оценки `score`.

Что смотреть в выводе:

- `status=ready` — метод отработал;
- `seconds` — сколько времени занял расчёт.

Для rule-based обычно всё быстро. Для `bi_encoder_zero_shot` может быть дольше, потому что загружается модель и считаются векторы текстов.


In [6]:
scored_methods: dict[str, dict[str, object]] = {}
skipped_methods: list[dict[str, str]] = []

if labeled_pairs.empty:
    display(pd.DataFrame([{"method": "not_available_yet", "status": labeling_status.loc[0, "status"], "seconds": 0.0}]))
else:
    scoring_rows = []
    for matcher in matchers:
        started = time.perf_counter()
        scores, status = _score_matcher(matcher, labeled_pairs)
        elapsed = time.perf_counter() - started
        scoring_rows.append({"method": matcher.name, "status": status, "seconds": round(elapsed, 3)})
        if status != "ready":
            skipped_methods.append({"method": matcher.name, "status": status})
            continue
        scored_methods[matcher.name] = {"matcher": matcher, "scores": scores, "seconds": elapsed}
    display(pd.DataFrame(scoring_rows))
    if skipped_methods:
        display(pd.DataFrame(skipped_methods))


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5282.57it/s]

,method,status,seconds
0,rule_based_fuzzy,ready,0.039
1,bi_encoder_zero_shot,ready,21.303


## Блок кода 7. Быстрый score sanity-check

Эта ячейка не выбирает threshold. Она только показывает диапазон score по методам, чтобы сразу увидеть пустые/битые прогоны.


In [ ]:
score_overview_rows: list[dict[str, object]] = []

for method, payload in scored_methods.items():
    score_series = pd.to_numeric(pd.Series(payload["scores"]), errors="coerce").dropna()
    score_overview_rows.append(
        {
            "method": method,
            "pairs": len(payload["scores"]),
            "score_min": float(score_series.min()) if not score_series.empty else None,
            "score_median": float(score_series.median()) if not score_series.empty else None,
            "score_max": float(score_series.max()) if not score_series.empty else None,
            "seconds": round(float(payload.get("seconds", 0.0)), 3),
        }
    )

if score_overview_rows:
    display(pd.DataFrame(score_overview_rows))
else:
    display(pd.DataFrame([{"method": "not_available_yet", "pairs": 0}]))


## Блок кода 8. Threshold evaluation выполняется после scoring всех моделей

Раньше здесь подбирался отдельный threshold по старой логике. Этот блок намеренно убран из основной логики: сначала ниже считаются cross-encoder/reranker scores, затем единый финальный block подбирает `threshold_same` для всех methods одинаково.


In [ ]:
binary_threshold_summary = pd.DataFrame()
binary_threshold_predictions = pd.DataFrame()

print("Threshold selection moved to the final binary benchmark block.")


## Блок кода 9. Матрицы ошибок больше не нужны отдельным block-ом

Финальный binary benchmark считает `precision`, `recall`, `f1`, `accuracy`, `false_merge_count`, `false_split_count` и cost напрямую для каждой threshold strategy.


In [ ]:
print("Binary metrics will be displayed after threshold_same selection.")


## Блок кода 10. Сохранение результатов перенесено в финальный benchmark block

Финальный block сохраняет только компактные отчёты:

- `artifacts/reports/binary_threshold_summary.csv` — одна строка на `method + split + threshold_strategy`;
- `artifacts/reports/binary_threshold_predictions.csv` — предсказания по парам для выбранных стратегий;
- `artifacts/reports/binary_threshold_by_volume_bucket.csv` — breakdown по объёму продаж, только если weighted-метрики доступны.

Raw predictions и старые `matching_*` CSV здесь не перезаписываются.


In [ ]:
print("Compact binary reports will be written after all selected methods are scored.")


## Что делать после этой тетрадки

Если результат `rule_based_fuzzy` и `bi_encoder_zero_shot` слабый, это нормально для первого прогона. Эта тетрадка нужна не для финальной победы, а чтобы увидеть нижнюю планку и понять, где методы ошибаются.

Главный вывод сейчас: похожесть текста сама по себе часто путает разные вкусы и типы соусов. Следующий разумный шаг — улучшать scorer/reranker на hard negatives, но threshold benchmark ниже остаётся forced binary: без manual review, LLM-review и triage.


## Новая итерация: добавляем готовый cross-encoder

Предыдущие блоки показали важную проблему: простая похожесть названий и обычные векторы часто путают похожие, но разные товары.

Теперь добавляем следующий метод из архитектуры: cross-encoder. Он читает пару товаров вместе: товар A и товар B одновременно. Поэтому он теоретически должен лучше замечать различия вроде `сырный` против `барбекю` или `сальса` против `сладкий чили`.

Пока это не обученная на наших данных модель, а готовая модель из `sentence-transformers`. Поэтому это промежуточный опыт: проверяем, помогает ли более внимательное сравнение пары даже без дообучения.


## Блок кода 11. Настройки cross-encoder

Эта ячейка добавляет новый метод, но не трогает старые результаты выше.

Что важно:

- `DEDUP_RUN_CROSS_ENCODER=0` можно поставить, если нужно временно пропустить этот блок.
- `DEDUP_CROSS_ENCODER_MODEL` теперь принимает alias из model registry или прямой Hugging Face model id.
- По умолчанию используется alias `cross_encoder_mmarco`, который указывает на `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`.
- Скачивание и повторное использование модели управляется через `ModelManager`: кэш `DEDUP_MODEL_CACHE_DIR`, offline-флаг `DEDUP_MODEL_LOCAL_ONLY=1`.

Если модель ещё не скачана, первый запуск может занять время. Если интернет недоступен, включи offline-режим только после предварительного прогрева кэша.


In [ ]:
from research.dedup import CrossEncoderMatcher
from research.dedup.matchers.cross_encoder import CrossEncoderConfig

RUN_CROSS_ENCODER = os.environ.get("DEDUP_RUN_CROSS_ENCODER", "1") == "1"
CROSS_ENCODER_MODEL = os.environ.get("DEDUP_CROSS_ENCODER_MODEL", "cross_encoder_mmarco")
CROSS_ENCODER_MODEL_SPEC = MODEL_MANAGER.resolve(CROSS_ENCODER_MODEL, backend=CROSS_ENCODER_BACKEND)
CROSS_ENCODER_BATCH_SIZE = int(
    os.environ.get("DEDUP_CROSS_ENCODER_BATCH_SIZE", str(CROSS_ENCODER_MODEL_SPEC.batch_size or 16))
)

print(f"Run cross-encoder: {RUN_CROSS_ENCODER}")
print(f"Cross-encoder model alias/input: {CROSS_ENCODER_MODEL}")
print(f"Cross-encoder model id: {CROSS_ENCODER_MODEL_SPEC.model_name}")
print(f"Cross-encoder batch size: {CROSS_ENCODER_BATCH_SIZE}")
print(f"Model cache dir: {MODEL_MANAGER.cache_dir}")
print(f"Local-only model loading: {MODEL_MANAGER.local_files_only}")


## Блок кода 12. Запуск cross-encoder на тех же парах

Эта ячейка считает `score` для каждой размеченной пары.

`score` здесь означает: насколько готовая модель считает пару подходящей для связи. У этой модели score не обязан быть от 0 до 1: важен не абсолютный смысл числа, а то, как он разделяет same-base и different-product пары. В финальном block этот score проходит через одно правило `score >= threshold_same`.

Мы используем те же `dev` и `test`, что выше. Это важно: новый метод сравнивается на той же отложенной части, а не на новых случайных строках.


In [12]:
cross_encoder_payload: dict[str, object] | None = None
cross_encoder_status_rows: list[dict[str, object]] = []

if not RUN_CROSS_ENCODER:
    cross_encoder_status_rows.append({"method": "cross_encoder_zero_shot", "status": "skipped_by_env", "seconds": 0.0})
elif labeled_pairs.empty:
    cross_encoder_status_rows.append({"method": "cross_encoder_zero_shot", "status": "skipped_empty_gold_set", "seconds": 0.0})
else:
    cross_encoder = CrossEncoderMatcher(
        CrossEncoderConfig(
            model_name=CROSS_ENCODER_MODEL,
            batch_size=CROSS_ENCODER_BATCH_SIZE,
        )
    )
    started = time.perf_counter()
    cross_scores, cross_status = _score_matcher(cross_encoder, labeled_pairs)
    elapsed = time.perf_counter() - started
    cross_encoder_status_rows.append(
        {"method": cross_encoder.name, "status": cross_status, "seconds": round(elapsed, 3)}
    )
    if cross_status == "ready":
        cross_encoder_payload = {"matcher": cross_encoder, "scores": cross_scores, "seconds": elapsed}

cross_encoder_status_df = pd.DataFrame(cross_encoder_status_rows)
display(cross_encoder_status_df)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6401.09it/s]

,method,status,seconds
0,cross_encoder_zero_shot,ready,27.663


## Блок кода 13. Cross-encoder score готовится для общей оценки

Эта ячейка больше не подбирает отдельный macro-F1 threshold. Если cross-encoder успешно посчитан, он попадёт в финальный binary benchmark вместе с остальными methods.


In [ ]:
if cross_encoder_payload is None:
    display(pd.DataFrame([{"method": "cross_encoder_zero_shot", "status": "not_available"}]))
else:
    score_series = pd.to_numeric(pd.Series(cross_encoder_payload["scores"]), errors="coerce").dropna()
    display(pd.DataFrame([
        {
            "method": cross_encoder_payload["matcher"].name,
            "status": "ready",
            "pairs": len(cross_encoder_payload["scores"]),
            "score_min": float(score_series.min()) if not score_series.empty else None,
            "score_median": float(score_series.median()) if not score_series.empty else None,
            "score_max": float(score_series.max()) if not score_series.empty else None,
            "seconds": round(float(cross_encoder_payload.get("seconds", 0.0)), 3),
        }
    ]))


Binary evaluation ниже покажет, уменьшает ли cross-encoder false merges при одном dev-selected `threshold_same`.


In [ ]:
print("Cross-encoder metrics will be displayed in the final binary benchmark block.")


## Блок кода 15. CSV выполняется финальным binary benchmark block

Cross-encoder не пишет отдельные calibrated CSV: все methods сохраняются единым набором compact-артефактов после общей оценки.


In [ ]:
print("CSV export is handled by the final binary benchmark block.")


## Что мы проверяем этой новой итерацией

Эта секция отвечает на конкретный вопрос: помогает ли готовый cross-encoder как более внимательная проверка пары.

Если он заметно снижает false merges на `test`, это сильный аргумент двигаться в сторону cross-encoder rerank.

Если он не помогает достаточно, это тоже нормальный результат: тогда показываем жюри, что готовой модели мало, и нужен следующий шаг — дообучение на наших парах или улучшение scorer/fusion на hard negatives.


## Общий бенчмарк всех matching-моделей

Эта секция сравнивает текущие baseline-методы и выбранные reranker-модели на одном и том же наборе размеченных пар:

- `rule_based_fuzzy`;
- `bi_encoder_zero_shot`;
- `cross_encoder_zero_shot`;
- `BAAI/bge-reranker-v2-m3` по умолчанию;
- опционально `Qwen/Qwen3-Reranker-4B`, `Qwen/Qwen3-Reranker-0.6B` и `jinaai/jina-reranker-v3`.

Запуск обычный: меняете настройки в следующей ячейке и жмёте Run. Никакие переменные окружения для включения блока не нужны.

Первый запуск может быть долгим именно на скачивании модели из Hugging Face. `Qwen/Qwen3-Reranker-4B` тяжёлый и на Mac может не помещаться в MPS, поэтому registry запускает его на CPU. Если нужен быстрый Qwen-smoke, попробуйте alias `qwen3_0_6b`.

## Блок кода 16. Настройки общего бенчмарка

Главное место для настройки моделей — верх следующей code-ячейки, блок `НАСТРОЙКИ ПОЛЬЗОВАТЕЛЯ`.

Туда вписываются новые reranker-модели, которые добавляются к уже посчитанным выше baseline-методам. Старые методы (`rule_based_fuzzy`, `bi_encoder_zero_shot`, `cross_encoder_zero_shot`) перечислять не нужно: они подтягиваются из предыдущих блоков этой же тетрадки автоматически.

Что менять чаще всего:

- `MY_RERANKER_MODELS` — список моделей для запуска. Можно писать короткие alias-ы (`"bge_m3"`, `"qwen3_4b"`, `"jina_v3"`) или полные Hugging Face model ids (`"BAAI/bge-reranker-v2-m3"`).
- `MY_RERANKER_MAX_PAIRS` — размер быстрого среза. `120` удобно для первого прогона, `0` означает весь размеченный gold-set.
- `MY_CUSTOM_RERANKER_BACKEND` — backend для model id, которого ещё нет в registry. Для обычных Hugging Face cross-encoder моделей оставляйте `CROSS_ENCODER_BACKEND`.

Переменные окружения `DEDUP_RERANKER_BENCHMARK_MODELS`, `DEDUP_RERANKER_BENCHMARK_MAX_PAIRS` и `DEDUP_RERANKER_BENCHMARK_BACKEND` нужны только для запуска из терминала; если они заданы, они переопределяют значения из code-ячейки.

После запуска ячейка показывает таблицу: что вы попросили, какой alias реально резолвится, какой model id будет скачан, какой backend используется и где лежит cache.


In [ ]:
from research.dedup import CrossEncoderMatcher, JinaRerankerMatcher
from research.dedup.matchers.cross_encoder import CrossEncoderConfig
from research.dedup.matchers.jina_reranker import JinaRerankerConfig

# === НАСТРОЙКИ ПОЛЬЗОВАТЕЛЯ ===
# Сюда вписывайте свои reranker-модели для общего benchmark.
# Можно писать короткие aliases: "bge_m3", "qwen3_4b", "qwen3_0_6b", "jina_v3".
# Можно писать полные Hugging Face ids: "BAAI/bge-reranker-v2-m3", "Qwen/Qwen3-Reranker-4B".
MY_RERANKER_MODELS = [
    "bge_m3",
    # "qwen3_4b",  # 4B грузится на CPU, чтобы не падать с MPS out of memory.
    # "qwen3_0_6b",  # более лёгкий Qwen для быстрого smoke-прогона.
    # "jina_v3",
    # "cross-encoder/ms-marco-MiniLM-L6-v2",  # пример своего cross-encoder id
]

# 120 = быстрый пробный срез. Для первого CPU-запуска Qwen-4B можно поставить 12-30. 0 = весь размеченный gold-set.
MY_RERANKER_MAX_PAIRS = 120

# Для своих Hugging Face cross-encoder ids оставляйте CROSS_ENCODER_BACKEND.
# Для Jina-like моделей с методом .rerank используйте TRANSFORMERS_AUTO_MODEL_BACKEND.
# Если хотите запрещать неизвестные ids, поставьте None.
MY_CUSTOM_RERANKER_BACKEND = CROSS_ENCODER_BACKEND
# === КОНЕЦ НАСТРОЕК ПОЛЬЗОВАТЕЛЯ ===


_reranker_models_env = os.environ.get("DEDUP_RERANKER_BENCHMARK_MODELS")
if _reranker_models_env is None:
    RERANKER_BENCHMARK_MODELS = [str(model).strip() for model in MY_RERANKER_MODELS if str(model).strip()]
    RERANKER_BENCHMARK_MODELS_SOURCE = "notebook: MY_RERANKER_MODELS"
else:
    RERANKER_BENCHMARK_MODELS = [model.strip() for model in _reranker_models_env.split(",") if model.strip()]
    RERANKER_BENCHMARK_MODELS_SOURCE = "env: DEDUP_RERANKER_BENCHMARK_MODELS"

_reranker_max_pairs_env = os.environ.get("DEDUP_RERANKER_BENCHMARK_MAX_PAIRS")
if _reranker_max_pairs_env is None:
    RERANKER_BENCHMARK_MAX_PAIRS = int(MY_RERANKER_MAX_PAIRS)
    RERANKER_BENCHMARK_MAX_PAIRS_SOURCE = "notebook: MY_RERANKER_MAX_PAIRS"
else:
    RERANKER_BENCHMARK_MAX_PAIRS = int(_reranker_max_pairs_env)
    RERANKER_BENCHMARK_MAX_PAIRS_SOURCE = "env: DEDUP_RERANKER_BENCHMARK_MAX_PAIRS"

_custom_backend_env = os.environ.get("DEDUP_RERANKER_BENCHMARK_BACKEND")
CUSTOM_RERANKER_BACKEND = _custom_backend_env.strip() if _custom_backend_env is not None else MY_CUSTOM_RERANKER_BACKEND
if isinstance(CUSTOM_RERANKER_BACKEND, str) and CUSTOM_RERANKER_BACKEND.strip().lower() in {"", "none", "null"}:
    CUSTOM_RERANKER_BACKEND = None

print(f"Models source: {RERANKER_BENCHMARK_MODELS_SOURCE}")
print(f"Requested reranker models: {RERANKER_BENCHMARK_MODELS or '[]'}")
print(f"Max pairs source: {RERANKER_BENCHMARK_MAX_PAIRS_SOURCE}; value={RERANKER_BENCHMARK_MAX_PAIRS or 'all'}")
print(f"Custom model backend: {CUSTOM_RERANKER_BACKEND or 'disabled'}")


def _fusion_from_model_spec(spec):
    threshold_high = spec.fusion_threshold_high if spec.fusion_threshold_high is not None else 0.5
    threshold_low = spec.fusion_threshold_low if spec.fusion_threshold_low is not None else 0.2
    return FusionConfig(threshold_high=threshold_high, threshold_low=threshold_low)


def _resolve_reranker_specs(model_inputs, *, custom_backend=None):
    specs = []
    errors = []
    input_by_alias = {}
    allowed_backends = {CROSS_ENCODER_BACKEND, TRANSFORMERS_AUTO_MODEL_BACKEND}
    if custom_backend not in allowed_backends | {None}:
        errors.append({"model_input": "MY_CUSTOM_RERANKER_BACKEND", "error": f"unsupported custom backend: {custom_backend}"})
        custom_backend = None

    for model_input in model_inputs:
        model_input = str(model_input).strip()
        if not model_input:
            continue
        try:
            spec = MODEL_MANAGER.resolve(model_input)
        except Exception as exc:
            if custom_backend is None:
                errors.append({
                    "model_input": model_input,
                    "error": f"{exc}; set MY_CUSTOM_RERANKER_BACKEND for custom model ids",
                })
                continue
            try:
                spec = MODEL_MANAGER.resolve(model_input, backend=custom_backend)
            except Exception as fallback_exc:
                errors.append({
                    "model_input": model_input,
                    "error": f"{exc}; custom backend {custom_backend}: {fallback_exc}",
                })
                continue
        if spec.backend not in allowed_backends:
            errors.append({"model_input": model_input, "error": f"unsupported backend for reranker benchmark: {spec.backend}"})
            continue
        specs.append(spec)
        input_by_alias[spec.alias] = model_input
    return specs, errors, input_by_alias


reranker_model_specs, reranker_model_errors, reranker_model_inputs = _resolve_reranker_specs(
    RERANKER_BENCHMARK_MODELS,
    custom_backend=CUSTOM_RERANKER_BACKEND,
)

benchmark_config_rows = [
    {
        "input": reranker_model_inputs.get(spec.alias, spec.alias),
        "alias": spec.alias,
        "method": spec.method_name or spec.alias,
        "backend": spec.backend,
        "model": spec.model_name,
        "batch_size": spec.batch_size or "",
        "device": spec.device or "auto",
        "documents_per_query": spec.documents_per_query or "",
        "max_pairs": RERANKER_BENCHMARK_MAX_PAIRS or "all",
        "cache_dir": str(MODEL_MANAGER.cache_dir),
        "local_only": MODEL_MANAGER.local_files_only,
    }
    for spec in reranker_model_specs
]
display(pd.DataFrame(benchmark_config_rows))
if reranker_model_errors:
    display(pd.DataFrame(reranker_model_errors))

reranker_benchmark_matchers = []
for spec in reranker_model_specs:
    method_name = spec.method_name or spec.alias
    if spec.backend == CROSS_ENCODER_BACKEND:
        reranker_benchmark_matchers.append(
            CrossEncoderMatcher(
                CrossEncoderConfig(
                    model_name=spec.alias,
                    method_name=method_name,
                    batch_size=spec.batch_size or 1,
                    device=spec.device,
                    trust_remote_code=spec.trust_remote_code,
                    prompts=spec.prompts,
                    default_prompt_name=spec.default_prompt_name,
                    fusion=_fusion_from_model_spec(spec),
                )
            )
        )
    elif spec.backend == TRANSFORMERS_AUTO_MODEL_BACKEND:
        reranker_benchmark_matchers.append(
            JinaRerankerMatcher(
                JinaRerankerConfig(
                    model_name=spec.alias,
                    method_name=method_name,
                    documents_per_query=spec.documents_per_query or 8,
                    trust_remote_code=spec.trust_remote_code,
                    fusion=_fusion_from_model_spec(spec),
                )
            )
        )

if not reranker_benchmark_matchers:
    display(pd.DataFrame([{
        "status": "no_models_selected",
        "hint": "add models to MY_RERANKER_MODELS, for example bge_m3, qwen3_4b or jina_v3",
    }]))
else:
    display(pd.DataFrame([
        {"method": matcher.name, "status": matcher.status().message}
        for matcher in reranker_benchmark_matchers
    ]))


## Блок кода 17. Запуск новых reranker-моделей

Эта ячейка считает score только для новых тяжёлых моделей из `MY_RERANKER_MODELS` после возможного env override. Старые методы выше уже посчитаны, поэтому здесь они не запускаются повторно.

Если `MY_RERANKER_MAX_PAIRS > 0`, берётся небольшой сбалансированный срез по `dev/test` и классам. Такой срез годится для проверки, что модель запускается. Для финального выбора лучшего решения поставьте `MY_RERANKER_MAX_PAIRS = 0` в предыдущей ячейке.


In [ ]:
def _benchmark_frame(frame: pd.DataFrame, max_pairs: int) -> pd.DataFrame:
    if frame.empty or max_pairs <= 0 or len(frame) <= max_pairs:
        return _with_benchmark_pair_key(frame)
    ordered = frame.copy()
    ordered["_round_robin_order"] = ordered.groupby(["eval_split", "same_base_product"]).cumcount()
    sampled = (
        ordered.sort_values(["_round_robin_order", "eval_split", "label"])
        .head(max_pairs)
        .drop(columns=["_round_robin_order"])
        .sort_index()
    )
    return _with_benchmark_pair_key(sampled.reset_index(drop=True))


reranker_benchmark_pairs = _benchmark_frame(labeled_pairs, RERANKER_BENCHMARK_MAX_PAIRS)
reranker_benchmark_payloads: dict[str, dict[str, object]] = {}
reranker_benchmark_status_rows: list[dict[str, object]] = []

if reranker_benchmark_pairs.empty:
    print("Benchmark skipped: gold-set is empty.")
else:
    print(f"Benchmark pairs: {len(reranker_benchmark_pairs)} / {len(labeled_pairs)}")
    for matcher in reranker_benchmark_matchers:
        started = time.perf_counter()
        scores, status = _score_matcher(matcher, reranker_benchmark_pairs)
        elapsed = time.perf_counter() - started
        reranker_benchmark_status_rows.append(
            {
                "method": matcher.name,
                "model": getattr(matcher.config, "model_name", ""),
                "status": status,
                "seconds": round(elapsed, 3),
                "pairs": len(reranker_benchmark_pairs),
                "seconds_per_pair": round(elapsed / len(reranker_benchmark_pairs), 4) if len(reranker_benchmark_pairs) else 0.0,
            }
        )
        if status == "ready":
            reranker_benchmark_payloads[matcher.name] = {
                "matcher": matcher,
                "scores": scores,
                "seconds": elapsed,
                "pairs": reranker_benchmark_pairs,
            }

if reranker_benchmark_status_rows:
    reranker_status_df = pd.DataFrame(reranker_benchmark_status_rows)
    display(reranker_status_df)
    ready_methods = reranker_status_df["status"].eq("ready").sum()
    status_text = "\n".join(reranker_status_df["status"].astype(str).tolist()).lower()
    if ready_methods == 0 and ("huggingface_hub" in status_text or "logging" in status_text):
        display(pd.DataFrame([{
            "problem": "reranker dependencies look stale or incompatible in the active kernel",
            "fix": f"Restart the Jupyter kernel, then run: {sys.executable} -m pip install -U -r requirements-research.txt",
            "why": "sentence-transformers / transformers could not import huggingface_hub.logging",
        }]))

## Блок кода 18. Binary threshold benchmark всех методов

Здесь собирается единая score-таблица для всех доступных methods и на `dev` выбирается один `threshold_same` для каждой стратегии:

- `threshold_max_f1` — максимальный обычный F1 на `dev`;
- `threshold_cost_sensitive` — минимальный `FP_COST * false_merge_count + FN_COST * false_split_count`;
- `threshold_max_weighted_f1` — максимальный weighted F1, если доступны объёмы продаж;
- `threshold_weighted_cost` — минимальный weighted cost, если доступны объёмы продаж.

`test` используется только для финальной проверки выбранных на `dev` порогов. Bucket cutoffs для объёма продаж тоже считаются только на `dev` и применяются к `test` без пересчёта.


In [ ]:
def _marketplace_key(value: object) -> str:
    if value is None:
        return "unknown_marketplace"
    try:
        if bool(value != value):
            return "unknown_marketplace"
    except TypeError:
        return "unknown_marketplace"
    text = str(value).casefold().strip()
    if not text:
        return "unknown_marketplace"
    return " ".join(text.split())


def _resolve_duckdb_path(project_root: Path) -> Path | None:
    env_path = os.environ.get("MPSTATS_DUCKDB_PATH")
    candidates = [Path(env_path).expanduser() if env_path else None, project_root / "mpstats.duckdb"]
    for candidate in candidates:
        if candidate is not None and candidate.exists():
            return candidate.resolve()
    return None


def _load_sales_volume_lookup() -> tuple[pd.DataFrame, str | None]:
    if not SALES_VOLUME_JOIN_ENABLED:
        return pd.DataFrame(), "weighted metrics disabled: sales volume join is disabled by DEDUP_ENABLE_SALES_VOLUME_JOIN=0"
    if importlib.util.find_spec("duckdb") is None:
        return pd.DataFrame(), "weighted metrics disabled: duckdb package is not available for sales volume join"
    db_path = _resolve_duckdb_path(PROJECT_ROOT)
    if db_path is None:
        return pd.DataFrame(), "weighted metrics disabled: mpstats.duckdb was not found and MPSTATS_DUCKDB_PATH is not set"

    import duckdb

    with duckdb.connect(str(db_path), read_only=True) as con:
        tables = con.execute("SHOW TABLES").fetchdf().iloc[:, 0].astype(str).tolist()
        if PRODUCTS_TABLE not in tables:
            return pd.DataFrame(), f"weighted metrics disabled: table {PRODUCTS_TABLE!r} not found in {db_path}"
        columns = set(con.execute(f"DESCRIBE {PRODUCTS_TABLE}").fetchdf()["column_name"].astype(str))
        required = {"Маркетплейс", "Артикул", SALES_VOLUME_COL}
        missing = sorted(required - columns)
        if missing:
            return pd.DataFrame(), f"weighted metrics disabled: missing columns in {PRODUCTS_TABLE}: {missing}"
        if "Категория" in columns:
            placeholders = ", ".join(["?"] * len(CATEGORY_ALIASES))
            raw = con.execute(
                f'SELECT "Маркетплейс", "Артикул", "{SALES_VOLUME_COL}" FROM {PRODUCTS_TABLE} '
                f'WHERE "Категория" IN ({placeholders})',
                CATEGORY_ALIASES,
            ).fetchdf()
        else:
            raw = con.execute(
                f'SELECT "Маркетплейс", "Артикул", "{SALES_VOLUME_COL}" FROM {PRODUCTS_TABLE}'
            ).fetchdf()

    raw["raw_record_id"] = raw["Маркетплейс"].map(_marketplace_key) + "::" + raw["Артикул"].astype(str).str.strip()
    raw["sales_volume"] = pd.to_numeric(raw[SALES_VOLUME_COL], errors="coerce").fillna(0.0).clip(lower=0)
    lookup = raw.groupby("raw_record_id", as_index=False)["sales_volume"].sum()
    if lookup.empty or lookup["sales_volume"].notna().sum() == 0:
        return pd.DataFrame(), "weighted metrics disabled: sales volume lookup is empty"
    return lookup, None


def _attach_sales_volumes(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    if frame.empty:
        return frame, pd.DataFrame([{"weight_status": "skipped_empty_scores"}])
    if detect_sales_volume_columns(frame) is not None:
        return frame, pd.DataFrame([{"weight_status": "using_existing_sales_volume_columns"}])
    if not {"raw_record_id_a", "raw_record_id_b"}.issubset(frame.columns):
        return frame, pd.DataFrame([{"weight_status": "unit_weight_fallback", "warning": "raw_record_id_a/raw_record_id_b are missing; no reliable sales volume join"}])

    lookup, warning = _load_sales_volume_lookup()
    if warning:
        return frame, pd.DataFrame([{"weight_status": "unit_weight_fallback", "warning": warning}])

    output = frame.copy()
    left_lookup = lookup.rename(columns={"raw_record_id": "raw_record_id_a", "sales_volume": "sales_volume_a"})
    right_lookup = lookup.rename(columns={"raw_record_id": "raw_record_id_b", "sales_volume": "sales_volume_b"})
    output = output.merge(left_lookup, on="raw_record_id_a", how="left")
    output = output.merge(right_lookup, on="raw_record_id_b", how="left")
    matched_left = int(output["sales_volume_a"].notna().sum())
    matched_right = int(output["sales_volume_b"].notna().sum())
    if matched_left == 0 and matched_right == 0:
        return frame, pd.DataFrame([{"weight_status": "unit_weight_fallback", "warning": "sales volume join found no matching raw_record_id keys"}])
    return output, pd.DataFrame([
        {
            "weight_status": "joined_sales_volume_from_mpstats_products",
            "db_path": str(_resolve_duckdb_path(PROJECT_ROOT)),
            "matched_left_rows": matched_left,
            "matched_right_rows": matched_right,
            "total_rows": len(output),
        }
    ])


def _load_legacy_score_frames(existing_methods: set[str]) -> list[pd.DataFrame]:
    legacy_paths = [
        DATA_DIR / "all_model_benchmark_predictions_sauces.csv",
        DATA_DIR / "matching_predictions_sauces.csv",
        DATA_DIR / "reranker_benchmark_predictions_sauces.csv",
    ]
    frames: list[pd.DataFrame] = []
    for path in legacy_paths:
        if not path.exists():
            continue
        legacy = pd.read_csv(path)
        required = {"method", "score", "eval_split"}
        if not required.issubset(legacy.columns):
            continue
        legacy = legacy[~legacy["method"].astype(str).isin(existing_methods)].copy()
        if legacy.empty:
            continue
        if "benchmark_source" not in legacy.columns:
            legacy["benchmark_source"] = "legacy_score_file"
        else:
            legacy["benchmark_source"] = legacy["benchmark_source"].fillna("legacy_score_file")
        legacy["benchmark_source"] = legacy["benchmark_source"].astype(str) + f"; restored_from={path.name}"
        frames.append(legacy)
        existing_methods.update(legacy["method"].astype(str).unique().tolist())
    return frames


def _collect_all_method_scores() -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    reranker_pairs = globals().get("reranker_benchmark_pairs", pd.DataFrame())
    reranker_payloads = globals().get("reranker_benchmark_payloads", {})
    baseline_payloads = globals().get("scored_methods", {})
    cross_payload = globals().get("cross_encoder_payload", None)
    benchmark_keys = (
        set(reranker_pairs["benchmark_pair_key"])
        if not reranker_pairs.empty and "benchmark_pair_key" in reranker_pairs.columns
        else set()
    )

    for method, payload in baseline_payloads.items():
        frame = _score_frame(method, labeled_pairs, payload["scores"], benchmark_source="baseline_or_embedding")
        frames.append(frame)

    if cross_payload is not None:
        matcher = cross_payload["matcher"]
        frames.append(
            _score_frame(matcher.name, labeled_pairs, cross_payload["scores"], benchmark_source="cross_encoder")
        )

    for method, payload in reranker_payloads.items():
        frames.append(
            _score_frame(method, payload["pairs"], payload["scores"], benchmark_source="reranker")
        )

    existing_methods = {
        str(method)
        for frame in frames
        if "method" in frame.columns
        for method in frame["method"].dropna().unique()
    }
    frames.extend(_load_legacy_score_frames(existing_methods))

    if not frames:
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True)
    if benchmark_keys:
        combined = combined[combined["benchmark_pair_key"].isin(benchmark_keys)].copy()
    return combined.reset_index(drop=True)


all_method_scores = _collect_all_method_scores()
if all_method_scores.empty:
    threshold_results = {
        "summary": pd.DataFrame(),
        "predictions": pd.DataFrame(),
        "weights_available": False,
        "weight_source": "unit_weight_fallback",
        "weight_warning": "no ready method scores",
    }
    binary_report_paths = write_binary_threshold_reports(threshold_results, REPORTS_DIR)
    display(pd.DataFrame([{"status": "no_ready_method_scores"}]))
    print(f"Saved binary summary: {binary_report_paths['summary']}")
    print(f"Saved binary predictions: {binary_report_paths['predictions']}")
else:
    all_method_scores, weight_join_status = _attach_sales_volumes(all_method_scores)
    display(weight_join_status)

    threshold_results = calibrate_and_evaluate_methods(all_method_scores, config=THRESHOLD_CONFIG)
    binary_threshold_summary = threshold_results["summary"].sort_values(
        ["method", "split", "threshold_strategy"],
        ascending=[True, True, True],
    ).reset_index(drop=True)
    binary_threshold_predictions = threshold_results["predictions"].copy()

    if threshold_results.get("weight_warning"):
        display(pd.DataFrame([{"warning": threshold_results["weight_warning"]}]))

    binary_report_paths = write_binary_threshold_reports(threshold_results, REPORTS_DIR)

    test_columns = [
        "method",
        "threshold_strategy",
        "threshold_same",
        "precision",
        "recall",
        "f1",
        "false_merge_count",
        "false_split_count",
        "cost",
        "weighted_f1",
        "weighted_total_cost",
        "weight_source",
    ]
    test_table = binary_threshold_summary[binary_threshold_summary["split"].eq("test")][test_columns].copy()
    if test_table.empty:
        display(pd.DataFrame([{"status": "no_test_split_available"}]))
    else:
        display(test_table.sort_values(["method", "threshold_strategy"]).reset_index(drop=True))

    print(f"Saved binary summary: {binary_report_paths['summary']}")
    print(f"Saved binary predictions: {binary_report_paths['predictions']}")
    if "by_volume_bucket" in binary_report_paths:
        print(f"Saved volume bucket report: {binary_report_paths['by_volume_bucket']}")


## Как читать общий бенчмарк

Главный файл теперь `artifacts/reports/binary_threshold_summary.csv`.

В нём одна строка на `method + split + threshold_strategy`:

- `threshold_max_f1` — обычный F1 на `dev`;
- `threshold_cost_sensitive` — минимум `5 * false_merge + 1 * false_split` на `dev`;
- `threshold_max_weighted_f1` и `threshold_weighted_cost` появляются только при доступных объёмах продаж.

`binary_threshold_predictions.csv` содержит предсказания по парам: score, true label, predicted label, `false_merge`, `false_split`, `pair_weight` и контекст пары.

Если weighted-метрики доступны, `binary_threshold_by_volume_bucket.csv` показывает breakdown по bucket объёма продаж: `zero / low / medium / high`. Cutoffs считаются только на `dev`, затем применяются к `test`.

Test split не используется для выбора threshold, bucket cutoffs или модели. Он нужен только для финальной проверки выбранных на dev порогов.
